# Fine-Tuning Representation Models for Classification
## Supervised Classification
- We'll fine-tune a pretrained BERT model to create a task-specific model similar to the one we used in chapter2
- Compared to the embedding model approach, we will fine-tune both the representation model and the classification head as a single architecture.
- To do so, instead of freezing the model, we allow it to be trainable and update its parameters during training. 
- As illustrated in img3, we will use a pretrained BERT model and add a neural network as a classification head, both of which will be fine-tuned for classification.

### Fine-Tuning a Pretrained BERT Model

- We will be using the same dataset we used in Chapter 4 to fine-tune our model, namely the Rotten Tomatoes dataset, which contains 5,331 positive and 5,331 negative movie reviews from Rotten Tomatoes



In [1]:
pip install "datasets>=2.18.0,<3" transformers>=4.38.2 sentence-transformers>=2.5.1 setfit>=1.0.3 accelerate>=0.27.2 seqeval>=1.2.2

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datamapplot 0.7.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
trl 1.2.0 requires datasets>=4.7.0, but you have datasets 2.21.0 which is incompatible.


In [18]:
# Step 1: Clear accelerator-managed resources
#trainer.accelerator.clear()

# Step 2: Delete Python objects
#del trainer, embedding_model

# Step 3: Force garbage collection
import gc
gc.collect()

# Step 4: Empty CUDA cache
import torch
torch.cuda.empty_cache()

#### Data

In [19]:
from datasets import load_dataset
toamatoes = load_dataset("rotten_tomatoes")
train_data, test_data = toamatoes["train"], toamatoes["test"]


#### Supervised Classification
##### HuggingFace Trainer
- The first step in our classification task is to select the underlying model we want to use. We use "bert-base-cased", which was pretrained on the English Wikipedia as well as a large dataset consisting of unpublished books.
- We define the number of labels that we want to predict beforehand. This is necessary to create the feedforward neural network that is applied on top of our pretrained model:

In [20]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load Model and Tokenizer
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4853.90it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trai

Next, we will tokenize our data:

In [21]:
from transformers import DataCollatorWithPadding

# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

# Tokenize train/test data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

Map: 100%|██████████| 1066/1066 [00:00<00:00, 15558.84 examples/s]


- Before creating the Trainer, we will want to prepare a special DataCollator. A DataCollator is a class that helps us build batches of data but also allows us to apply data augmentation.
- During this process of tokenization, and as shown in Chapter 9, we will add padding to the input text to create equally sized representations. We use DataCollatorWithPadding for that.

In [22]:
import numpy as np
import evaluate

def compute_metrics(eval_pred):
    """Compute F1 score"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    load_f1 = evaluate.load("f1")
    f1= load_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"f1": f1}

Next, we instantiate our Trainer:
- The TrainingArguments class defines hyperparameters we want to tune, such as the learning rate and how many epochs (rounds) we want to train. 
- The Trainer is used to execute the training process.

In [27]:


from transformers import TrainingArguments, Trainer

# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   eval_strategy="epoch",
   report_to="none"
)

# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)



In [28]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.197280,0.489843,0.842204


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.99s/it]


TrainOutput(global_step=534, training_loss=0.20801622769359346, metrics={'train_runtime': 68.7876, 'train_samples_per_second': 124.005, 'train_steps_per_second': 7.763, 'total_flos': 227605451772240.0, 'train_loss': 0.20801622769359346, 'epoch': 1.0})

We get an F1 score of 0.85, which is quite a bit higher than the task-specific model we used in Chapter 4, which resulted in an F1 score of 0.80. It shows that fine-tuning a model yourself can be more advantageous than using a pretrained model. It only costs us a couple of minutes to train.

### Freeze Layers
- To further showcase the importance of training the entire network, the next example will demonstrate how you can use Hugging Face Transformers to freeze certain layers of your network.
- We will freeze the main BERT model and allow only updates to pass through the classification head. This will be a great comparison as we will keep everything the same, except for freezing specific layers.

In [31]:
import gc
gc.collect()

# Step 4: Empty CUDA cache
import torch
torch.cuda.empty_cache()

In [32]:
# Load the model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3315.25it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trai

In [33]:
for name, param in model.named_parameters():
    print(name)

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

Our pretrained BERT model contains a lot of layers that we can potentially freeze. Inspecting these layers gives insight into the structure of the network and what we might want to freeze:

In [37]:
# We can check whether the model was correctly updated
for name, param in model.named_parameters():
     print(f"Parameter: {name} ----- {param.requires_grad}")

Parameter: bert.embeddings.word_embeddings.weight ----- True
Parameter: bert.embeddings.position_embeddings.weight ----- True
Parameter: bert.embeddings.token_type_embeddings.weight ----- True
Parameter: bert.embeddings.LayerNorm.weight ----- True
Parameter: bert.embeddings.LayerNorm.bias ----- True
Parameter: bert.encoder.layer.0.attention.self.query.weight ----- True
Parameter: bert.encoder.layer.0.attention.self.query.bias ----- True
Parameter: bert.encoder.layer.0.attention.self.key.weight ----- True
Parameter: bert.encoder.layer.0.attention.self.key.bias ----- True
Parameter: bert.encoder.layer.0.attention.self.value.weight ----- True
Parameter: bert.encoder.layer.0.attention.self.value.bias ----- True
Parameter: bert.encoder.layer.0.attention.output.dense.weight ----- True
Parameter: bert.encoder.layer.0.attention.output.dense.bias ----- True
Parameter: bert.encoder.layer.0.attention.output.LayerNorm.weight ----- True
Parameter: bert.encoder.layer.0.attention.output.LayerNorm.bia

We could choose to only freeze certain layers to speed up computing but still allow the main model to learn from the classification task. Generally, we want frozen layers to be followed by trainable layers.

In [38]:
for name, param in model.named_parameters():

     # Trainable classification head
     if name.startswith("classifier"):
        param.requires_grad = True

      # Freeze everything else
     else:
        param.requires_grad = False

In [39]:
# We can check whether the model was correctly updated
for name, param in model.named_parameters():
     print(f"Parameter: {name} ----- {param.requires_grad}")



Parameter: bert.embeddings.word_embeddings.weight ----- False
Parameter: bert.embeddings.position_embeddings.weight ----- False
Parameter: bert.embeddings.token_type_embeddings.weight ----- False
Parameter: bert.embeddings.LayerNorm.weight ----- False
Parameter: bert.embeddings.LayerNorm.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.query.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.query.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.key.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.key.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.value.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.value.bias ----- False
Parameter: bert.encoder.layer.0.attention.output.dense.weight ----- False
Parameter: bert.encoder.layer.0.attention.output.dense.bias ----- False
Parameter: bert.encoder.layer.0.attention.output.LayerNorm.weight ----- False
Parameter: bert.encoder.layer.0.attention.output

In [40]:
# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

In [41]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.697492,0.686295,0.602243


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.11s/it]


TrainOutput(global_step=534, training_loss=0.6965191230345308, metrics={'train_runtime': 28.6079, 'train_samples_per_second': 298.17, 'train_steps_per_second': 18.666, 'total_flos': 227605451772240.0, 'train_loss': 0.6965191230345308, 'epoch': 1.0})

F1 score is 0.60, which is quite a bit lower compared to our original 0.85 score. Instead of freezing nearly all layers, let’s freeze everything up until encoder block 10 as illustrated in Figure 11-6, and see how it affects performance. A major benefit is that this reduces computation but still allows updates to flow through part of the pretrained model:

### Freeze blocks 1-5

In [42]:
# We can check whether the model was correctly updated
for index, (name, param) in enumerate(model.named_parameters()):
     print(f"Parameter: {index}{name} ----- {param.requires_grad}")

Parameter: 0bert.embeddings.word_embeddings.weight ----- False
Parameter: 1bert.embeddings.position_embeddings.weight ----- False
Parameter: 2bert.embeddings.token_type_embeddings.weight ----- False
Parameter: 3bert.embeddings.LayerNorm.weight ----- False
Parameter: 4bert.embeddings.LayerNorm.bias ----- False
Parameter: 5bert.encoder.layer.0.attention.self.query.weight ----- False
Parameter: 6bert.encoder.layer.0.attention.self.query.bias ----- False
Parameter: 7bert.encoder.layer.0.attention.self.key.weight ----- False
Parameter: 8bert.encoder.layer.0.attention.self.key.bias ----- False
Parameter: 9bert.encoder.layer.0.attention.self.value.weight ----- False
Parameter: 10bert.encoder.layer.0.attention.self.value.bias ----- False
Parameter: 11bert.encoder.layer.0.attention.output.dense.weight ----- False
Parameter: 12bert.encoder.layer.0.attention.output.dense.bias ----- False
Parameter: 13bert.encoder.layer.0.attention.output.LayerNorm.weight ----- False
Parameter: 14bert.encoder.laye

In [44]:
# Load model
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Encoder block 10 starts at index 165 and
# we freeze everything before that block
for index, (name, param) in enumerate(model.named_parameters()):
    if index < 165:
        param.requires_grad = False

# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5687.52it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trai

In [45]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.460971,0.410286,0.818702


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.77s/it]


TrainOutput(global_step=534, training_loss=0.4568156981736087, metrics={'train_runtime': 33.3181, 'train_samples_per_second': 256.017, 'train_steps_per_second': 16.027, 'total_flos': 227605451772240.0, 'train_loss': 0.4568156981736087, 'epoch': 1.0})

We got an F1 score of 0.81, which is much higher than our previous score of 0.61 when freezing all layers. It demonstrates that although we generally want to train as many layers as possible, you can get away with training less if you do not have the necessary computing power.

In [ ]:
scores = []
for index in range(12):
    # Re-load model
    model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)
    tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

    # Freeze encoder blocks 0-index
    for name, param in model.named_parameters():
        if "layer" in name:
            layer_nr = int(name.split("layer")[1].split(".")[1])
            if layer_nr <= index:
                param.requires_grad = False
            else:
                param.requires_grad = True

#       # Train
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_train,
            eval_dataset=tokenized_test,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            )
        trainer.train()
        # Evaluate to get metrics including F1
        eval_results = trainer.evaluate()
        # Extract F1 score
        f1_score = eval_results['eval_f1']
        scores.append(f1_score)


        